# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and description
print("Available record sets in this dataset:\n")
for record_set in dataset.metadata.record_sets:
    print(f"@id: {record_set.id}\n  Name: {record_set.name}\n  Description: {record_set.description}\n")

# List fields for each record set
for record_set in dataset.metadata.record_sets:
    print(f"Fields for record set {record_set.id}:")
    for field in record_set.fields:
        print(f"  @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets
from collections import OrderedDict
record_set_ids = [record_set.id for record_set in dataset.metadata.record_sets]
dataframes = OrderedDict()

for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns for {rs_id}: {df.columns.tolist()}\n")

# Display the first few rows of the first record set
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f'Record set chosen for EDA: {main_record_set_id}')
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis from the main record set
main_df = dataframes[main_record_set_id]
numeric_fields = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]

# Choose the first numeric field if available
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f'Numeric field selected for EDA: {numeric_field_id}')
    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notnull().any() else 0
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try grouping by a categorical field, if available
    categorical_fields = [col for col in main_df.columns if pd.api.types.is_string_dtype(main_df[col])]
    group_field = categorical_fields[0] if categorical_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Average {numeric_field_id} grouped by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric fields present in this record set to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if 'filtered_df' in locals() and not filtered_df.empty and numeric_fields:
    # Histogram of numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    # If a group field available, make a boxplot
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
        plt.xticks(rotation=75)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.